In [ ]:
!pip install "gymnasium[atari, accept-rom-license]"

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import cv2
import gymnasium as gym

# ─────────────────────────────────────────────
# Hyperparameters — exactly as stated in paper
# ─────────────────────────────────────────────

REPLAY_MEMORY_SIZE = 1_000_000
MINIBATCH_SIZE     = 32          
GAMMA              = 0.99        
EPS_START          = 1.0         
EPS_END            = 0.05        
EPS_ANNEAL_FRAMES  = 1_000_000
TOTAL_FRAMES       = 1_000_000
FRAME_SKIP         = 4


def preprocess(frame: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)   # 210×160
    resized = cv2.resize(gray, (84, 110))             # 110×84
    cropped = resized[18:102, :]                      # 84×84 (captures play area)
    return cropped.astype(np.float32) / 255.0


class FrameStack:
    """Stacks the last 4 frames to form φ(s)."""

    def __init__(self):
        self.frames = deque(maxlen=4)

    def reset(self, frame: np.ndarray):
        processed = preprocess(frame)
        for _ in range(4):
            self.frames.append(processed)
        return self._get()

    def step(self, frame: np.ndarray):
        self.frames.append(preprocess(frame))
        return self._get()

    def _get(self) -> np.ndarray:
        return np.stack(self.frames, axis=0)

class DQNNetwork(nn.Module):
    def __init__(self, num_actions: int):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 16, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=4, stride=2),
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 9 * 9, 256),
            nn.ReLU(),
            nn.Linear(256, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = x.flatten(start_dim=1)
        return self.fc(x)


class ReplayMemory:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def store(self, phi, action, reward, phi_next, terminal):
        self.buffer.append((phi, action, reward, phi_next, terminal))

    def sample(self, batch_size: int, device: torch.device):
        batch = random.sample(self.buffer, batch_size)
        phi, actions, rewards, phi_next, terminals = zip(*batch)
        return (
            torch.tensor(np.array(phi),      dtype=torch.float32, device=device),
            torch.tensor(actions,            dtype=torch.long,    device=device),
            torch.tensor(rewards,            dtype=torch.float32, device=device),
            torch.tensor(np.array(phi_next), dtype=torch.float32, device=device),
            torch.tensor(terminals,          dtype=torch.float32, device=device),
        )

    def __len__(self):
        return len(self.buffer)


def get_epsilon(frame: int) -> float:
    """Linearly anneal ε from 1.0 → 0.1 over first million frames, then fix."""
    if frame >= EPS_ANNEAL_FRAMES:
        return EPS_END
    return EPS_START - (EPS_START - EPS_END) * (frame / EPS_ANNEAL_FRAMES)


def train(env: gym.Env) -> DQNNetwork:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    Q = DQNNetwork(env.action_space.n).to(device)

    optimizer = optim.RMSprop(Q.parameters(), lr=25e-5, eps=0.01, alpha=0.95)

    D = ReplayMemory(REPLAY_MEMORY_SIZE)

    frame_stack = FrameStack()
    total_frames = 0
    episode = 0
    losses = []
    while total_frames < TOTAL_FRAMES:
        obs, _ = env.reset()
        phi = frame_stack.reset(obs)
        episode_reward = 0
        episode_losses = [] 
        episode += 1

        while True:
            eps = get_epsilon(total_frames)

            if random.random() < eps:
                action = env.action_space.sample()
            else:
                phi_t = torch.tensor(phi, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    action = Q(phi_t).argmax(dim=1).item()

            total_reward = 0.0
            for _ in range(FRAME_SKIP):
                obs, reward, terminated, truncated, _ = env.step(action)
                total_reward += reward
                if terminated or truncated:
                    break
            terminal = terminated or truncated

            phi_next = frame_stack.step(obs)

            D.store(phi, action, total_reward, phi_next, float(terminal))

            phi = phi_next
            episode_reward += total_reward
            total_frames += 1

            if len(D) >= MINIBATCH_SIZE:
                phis, actions_b, rewards_b, phis_next, terminals_b = D.sample(MINIBATCH_SIZE, device)

                with torch.no_grad():
                    max_next_q = Q(phis_next).max(dim=1).values
                    y = rewards_b + GAMMA * max_next_q * (1.0 - terminals_b)

                q_pred = Q(phis).gather(1, actions_b.unsqueeze(1)).squeeze(1)
                loss = nn.functional.mse_loss(q_pred, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                episode_losses.append(loss.item())

            if terminal:
                break
        losses.append(episode_losses)
        print(f"Episode {episode:4d} | Frames {total_frames:>9,} | "
                f"Reward {episode_reward:>7.1f} | ε {eps:.3f} | Loss {np.mean(episode_losses) if episode_losses else 0:.4f}")

    return Q

In [ ]:
import gymnasium as gym
import ale_py
env = gym.make("ALE/Alien-v5")
train(env)